# Supporting Figure S5

![Full Figure](../figures/suppS5_FULL.png)

## Code for Supporting Figure S5: training history and session structure

**Goal:** Relate trained violation rate to fixation training and show where violations occur within sessions, including edge-trimmed autocorrelation robustness for Figure 3.

| Panel | Content |
|---|---|
| a | Fixation-stage violation rate vs. trained violation rate (N = 16 animals) |
| b | Sessions in stage 1 vs. trained violation rate |
| c | Sessions to trained vs. trained violation rate |
| d | Violation rate across session progress (10 bins; all trials) |
| e | Violation rate aligned to session start (first 20 trials) |
| f | Violation rate aligned to session end (last 20 trials) |
| g | Full vs. edge-trimmed session autocorrelation (lags 1–20) |

**Outputs:** `figures/suppS5a_fixation_stage_violations.png`, `figures/suppS5b_time_in_fixation_stage.png`, `figures/suppS5c_time_to_trained.png`, `figures/suppS5d_session_progress.png`, `figures/suppS5e_session_start.png`, `figures/suppS5f_session_end.png`, `figures/suppS5g_autocorrelation_robustness.png`.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from violations_glm.config import VIOLATION_COLOR
from violations_glm.io import (
    load_autocorrelation_summary,
    load_history_robustness,
    load_session_edge_rates,
    load_session_position_statistics,
    load_session_progress_violation_rates,
    load_training_predictor_statistics,
    load_training_predictors,
)
from violations_glm.plotting import (
    animal_association_scatter,
    animal_palette,
    configure_publication_dpi,
    save_figure,
)

configure_publication_dpi()

sns.set_context(
    "notebook",
    rc={
        "font.size": 20,
        "axes.titlesize": 20,
        "axes.labelsize": 20,
        "axes.titleweight": "bold",
        "xtick.labelsize": 15,
        "ytick.labelsize": 15,
    },
)


In [ ]:
training_predictors = load_training_predictors()
training_statistics = load_training_predictor_statistics()
progress_rates = load_session_progress_violation_rates(include_edges=True)
position_statistics = load_session_position_statistics()
edge_rates = load_session_edge_rates()
palette = animal_palette()
animal_grays = ["gray"] * training_predictors["animal_id"].nunique()
trained_viol_label = "Trained Violation Rate (%)"


## Panel A — Fixation-stage violation rate


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
animal_association_scatter(
    ax,
    training_predictors,
    "fixation_stage_viol_rate",
    "trained_viol_rate",
    training_statistics["fixation_stage_viol_rate"],
    palette=palette,
)
ax.set(
    xlabel="Fix Grow Stage 1 Violation Rate (%)",
    ylabel=trained_viol_label,
    title="Fixation-Stage Violations",
    ylim=(0, 50),
    xlim=(0, 50),
)
sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, "suppS5a_fixation_stage_violations")


## Panel B — Time in fixation stage


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
animal_association_scatter(
    ax,
    training_predictors,
    "fixation_stage_sessions",
    "trained_viol_rate",
    training_statistics["fixation_stage_sessions"],
    palette=palette,
)
ax.set(
    xlabel="Sessions in Stage 1",
    ylabel=trained_viol_label,
    title="Time in Fixation Stage",
    ylim=(0, 50),
    xlim=(10, 40),
)
sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, "suppS5b_time_in_fixation_stage")


## Panel C — Time to trained


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
animal_association_scatter(
    ax,
    training_predictors,
    "sessions_to_trained",
    "trained_viol_rate",
    training_statistics["sessions_to_trained"],
    palette=palette,
)
ax.set(
    xlabel="Sessions to Trained",
    ylabel=trained_viol_label,
    title="Time to Trained",
    ylim=(0, 50),
)
sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, "suppS5c_time_to_trained")


## Panel D — Violation rate across session progress

Ten equal trial fractions per session using all trials. Gray lines: individual animals; colored line: mean ± SEM.


In [ ]:
all_stats = position_statistics["all_trials"]["slope_per_10_percent"]
fig, ax = plt.subplots(figsize=(6, 4))
sns.pointplot(
    data=progress_rates,
    x="session_progress",
    y="viol_rate",
    hue="animal_id",
    palette=animal_grays,
    alpha=0.35,
    native_scale=True,
    legend=False,
    ax=ax,
)
sns.pointplot(
    data=progress_rates,
    x="session_progress",
    y="viol_rate",
    color=VIOLATION_COLOR,
    errorbar="se",
    linewidth=4,
    native_scale=True,
    ax=ax,
)
ax.text(
    0.04,
    0.96,
    f"slope = {all_stats['mean_slope']:.2f} pp/10%\n"
    f"p = {all_stats['p_value']:.2g}\n"
    f"N = {all_stats['n_animals']}",
    transform=ax.transAxes,
    va="top",
)
ax.set(
    xticks=[5, 25, 45, 65, 85, 95],
    xlabel="Session Progress (%)",
    ylabel="Violation Rate (%)",
    title="Violation Rate Across Session",
    ylim=(0, 50),
)
sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, "suppS5d_session_progress")


## Panel E — Session start


In [ ]:
start = edge_rates.query("aligned_to == 'Start'")
fig, ax = plt.subplots(figsize=(6, 4))
sns.lineplot(
    data=start,
    x="trial_offset",
    y="viol_rate",
    units="animal_id",
    estimator=None,
    color="gray",
    alpha=0.4,
    linewidth=1,
    ax=ax,
)
sns.lineplot(
    data=start,
    x="trial_offset",
    y="viol_rate",
    color=VIOLATION_COLOR,
    errorbar="se",
    linewidth=4,
    ax=ax,
)
ax.set(
    xlabel="Trial from Session Start",
    ylabel="Violation Rate (%)",
    title="Session Start",
    ylim=(0, 100),
)
sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, "suppS5e_session_start")


## Panel F — Session end


In [ ]:
end = edge_rates.query("aligned_to == 'End'")
fig, ax = plt.subplots(figsize=(6, 4))
sns.lineplot(
    data=end,
    x="trial_offset",
    y="viol_rate",
    units="animal_id",
    estimator=None,
    color="gray",
    alpha=0.4,
    linewidth=1,
    ax=ax,
)
sns.lineplot(
    data=end,
    x="trial_offset",
    y="viol_rate",
    color=VIOLATION_COLOR,
    errorbar="se",
    linewidth=4,
    ax=ax,
)
ax.set(
    xlabel="Trial from Session End",
    ylabel="Violation Rate (%)",
    title="Session End",
    ylim=(0, 100),
)
sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, "suppS5f_session_end")


## Panel G — Autocorrelation robustness

Same permutation procedure as Figure 3a, with and without the first and last 10 trials of each session. Positive lags only.


In [ ]:
autocorr_full = load_autocorrelation_summary()
autocorr_trim = load_autocorrelation_summary(edges_trimmed=True)
history_robustness = load_history_robustness()

fig, ax = plt.subplots(figsize=(6, 4))
for table, color, label in (
    (autocorr_full, VIOLATION_COLOR, "Full sessions"),
    (autocorr_trim, "0.35", "Trimmed edges"),
):
    positive = table.loc[table["lag"] > 0]
    ax.plot(
        positive["lag"],
        positive["observed_mean"],
        color=color,
        marker="o",
        markersize=4,
        label=label,
    )
    ax.fill_between(
        positive["lag"],
        positive["null_ci_lower"],
        positive["null_ci_upper"],
        color=color,
        alpha=0.15,
    )
ax.set(xlabel="Lag (trials)", ylabel="Autocorrelation", xlim=(0.5, 20.5))
ax.legend(frameon=False)
sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, "suppS5g_autocorrelation_robustness")

sig_mask = (
    autocorr_full["significant"].fillna(False).astype(bool)
    & autocorr_full["lag"].between(1, 7)
)
sig_full = autocorr_full.loc[sig_mask, "lag"].astype(int).tolist()
sig_trim = autocorr_trim.loc[
    autocorr_trim["significant"].fillna(False).astype(bool)
    & autocorr_trim["lag"].between(1, 7),
    "lag",
].astype(int).tolist()
print("Significant lags 1–7 (full):", sig_full)
print("Significant lags 1–7 (trimmed):", sig_trim)
print(
    "Lag-1 observed:",
    float(autocorr_full.loc[autocorr_full.lag == 1, "observed_mean"].iloc[0]),
    "->",
    float(autocorr_trim.loc[autocorr_trim.lag == 1, "observed_mean"].iloc[0]),
)
pd.DataFrame(history_robustness).T
